# Preparación de datos — Explorador de Aleaciones de Alta Entropía

**Etapa 3 del proyecto · Ingeniería de Materiales · Dataset 3 (aleaciones multicomponentes)**

Este notebook toma `Dataset_3 - Dataset_3.csv`, repite la limpieza de la Etapa 1, agrega las clasificaciones de la Etapa 2
(familia de fase, sistema de aleación, unidad de análisis) y calcula parámetros de composición usados en el diseño de
aleaciones de alta entropía (VEC, diferencia de tamaño atómico δ y entropía de mezcla). También asigna a cada registro un
**indicador de calidad del dato** basado en los problemas detectados en las etapas anteriores.

Flujo: `Dataset_3 - Dataset_3.csv` → este notebook → `datos.js` → `index.html` (GitHub Pages)

## 1. Carga

In [1]:
import os, re, json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ruta = '/content/drive/MyDrive/Colab Notebooks/Dataset_3 - Dataset_3.csv'
if not os.path.exists(ruta):
    ruta = 'Dataset_3 - Dataset_3.csv'

df = pd.read_csv(ruta)
print('Dataset original:', df.shape)

Dataset original: (1545, 27)


## 2. Limpieza (criterios de la Etapa 1)

Las columnas numéricas que vienen como texto se separan en **valor**, **incertidumbre** y **tipo** (exacto, límite superior,
límite inferior o rango). Las etiquetas de fase y de procesamiento se normalizan. No se elimina ningún registro.

In [2]:
def parsear_valor(valor):
    """'168 $\\pm$ 5', '<160', '>50' o '1-2'  ->  (valor, incertidumbre, tipo)."""
    if pd.isna(valor):
        return (np.nan, np.nan, None)
    texto = str(valor).strip()
    m = re.match(r'^([\d.]+)\s*\$\\pm\$\s*([\d.]*)$', texto)
    if m:
        return (float(m.group(1)), float(m.group(2)) if m.group(2) else np.nan, 'exacto')
    if texto.startswith('<'):
        return (float(re.sub(r'[^\d.]', '', texto)), np.nan, 'limite_superior')
    if texto.startswith('>'):
        return (float(re.sub(r'[^\d.]', '', texto)), np.nan, 'limite_inferior')
    m = re.match(r'^([\d.]+)\s*-\s*([\d.]+)$', texto)
    if m:
        return ((float(m.group(1)) + float(m.group(2))) / 2, np.nan, 'rango')
    try:
        return (float(texto), np.nan, 'exacto')
    except ValueError:
        return (np.nan, np.nan, None)

COLS = {
    'HV': 'PROPERTY: HV', 'YS': 'PROPERTY: YS (MPa)', 'UTS': 'PROPERTY: UTS (MPa)',
    'El': 'PROPERTY: Elongation (%)', 'Elp': 'PROPERTY: Elongation plastic (%)',
    'Eexp': 'PROPERTY: Exp. Young modulus (GPa)', 'rho_exp': 'PROPERTY: Exp. Density (g/cm$^3$)',
    'grano': r'PROPERTY: grain size ($\mu$m)',
}
d = pd.DataFrame({
    'formula': df['FORMULA'].str.strip(),
    'doi': df['REFERENCE: doi'],
    'fases': df['PROPERTY: Type of phases'],
    'SM': df['PROPERTY: Single/Multiphase'],
    'proc': df['PROPERTY: synthesis method'].str.replace(' ', '', regex=False).str.upper(),
    'ensayo': df['PROPERTY: Type of test'],
    'T': df[r'PROPERTY: Test temperature ($^\circ$C)'],
    'rho_rom': df['PROPERTY: ROM Density (g/cm$^3$)'],
    'E_rom': df['PROPERTY: ROM Young modulus (GPa)'],
    'O': df['PROPERTY: O content (wppm)'],
    'C_wppm': df['PROPERTY: C content (wppm)'],
    'comentario': df['REFERENCE: comment'],
})
for corto, col in COLS.items():
    r = df[col].apply(parsear_valor)
    d[corto] = r.str[0]
    d[corto + '_inc'] = r.str[1]
    d[corto + '_tipo'] = r.str[2]

print('Tipos por columna:')
for corto in COLS:
    print(f'  {corto}:', d[corto + '_tipo'].value_counts().to_dict())

Tipos por columna:
  HV: {'exacto': 528, 'limite_superior': 2}
  YS: {'exacto': 1067}
  UTS: {'exacto': 539}
  El: {'exacto': 562, 'limite_inferior': 48, 'rango': 6, 'limite_superior': 3}
  Elp: {'exacto': 149}
  Eexp: {'exacto': 145}
  rho_exp: {'exacto': 112}
  grano: {'exacto': 223, 'rango': 13, 'limite_inferior': 1}


## 3. Clasificaciones de la Etapa 2

- **Familia de fase:** FCC, BCC (incluye B2), FCC+BCC, HCP u otra, según las redes reportadas. Las fases secundarias (Laves, sigma, intermetálicos, etc.) se marcan aparte.
- **Sistema de aleación:** *refractaria* si Nb, Mo, Ta, W, V, Hf, Zr, Re y Ti suman al menos 50 % atómico; *de metales 3d* si Co, Cr, Fe, Ni, Mn y Cu suman al menos 50 %; *mixta* en otro caso.
- **Lectura de fórmulas con paréntesis:** en la Etapa 2 el coeficiente de un paréntesis multiplicaba a cada elemento del grupo. En 32 de las 35 fórmulas con paréntesis los coeficientes del nivel superior suman 100 (o 1), es decir, están en porcentaje atómico: `Co25Cr25(FeNi)50` significa 25 % de Fe y 25 % de Ni, no 50 % de cada uno. Aquí se corrige esa lectura (ver `parsear_formula`) y se comprueba que la clasificación por sistema no cambia.
- **Unidad de análisis:** registros con la misma aleación, artículo, ensayo, temperatura y procesamiento. Si una unidad tiene varios registros con YS distinto, se marca, porque en la Etapa 2 se vio que no son réplicas.

In [3]:
def _leer(s, i=0):
    """Lee la fórmula desde la posición i. Devuelve una lista de (contenido, coeficiente) y la posición final.
    El contenido es un símbolo o, para un paréntesis, otra lista."""
    items = []
    while i < len(s) and s[i] != ')':
        if s[i] == '(':
            sub, i = _leer(s, i + 1)
            i += 1                      # cierra ')'
            m = re.match(r'\d*\.?\d+', s[i:])
            coef = float(m.group()) if m else 1.0
            i += len(m.group()) if m else 0
            items.append((sub, coef))
        else:
            m = re.match(r'([A-Z][a-z]?)(\d*\.?\d+)?', s[i:])
            if not m:
                raise ValueError(f'No se pudo leer {s[i:]!r}')
            items.append((m.group(1), float(m.group(2)) if m.group(2) else 1.0))
            i += len(m.group())
    return items, i

def _expandir(items, normalizar):
    moles = {}
    for cont, coef in items:
        if isinstance(cont, str):
            moles[cont] = moles.get(cont, 0) + coef
        else:
            sub = _expandir(cont, normalizar)
            total = sum(sub.values()) if normalizar else 1.0
            for e, n in sub.items():
                moles[e] = moles.get(e, 0) + coef * n / total
    return moles

def parsear_formula(formula):
    """{elemento: moles}. Si los coeficientes del nivel superior suman 100 o 1, la fórmula está en porcentaje o en
    fracción y cada paréntesis representa esa cantidad del total: '(CoCrFeNi)99Mn1' = 99 % de CoCrFeNi equiatómico
    y 1 % de Mn. Si no, el coeficiente multiplica literalmente: 'MoNbRe0.5W(TaC)0.2' = 0.2 Ta + 0.2 C."""
    items, _ = _leer(str(formula).replace(' ', ''))
    suma = sum(coef for _, coef in items)
    hay_grupos = any(not isinstance(c, str) for c, _ in items)
    normalizar = hay_grupos and (abs(suma - 100) < 0.5 or abs(suma - 1) < 0.005)
    return _expandir(items, normalizar)

REFRACTARIOS = {'Nb', 'Mo', 'Ta', 'W', 'V', 'Hf', 'Zr', 'Re', 'Ti'}
METALES_3D = {'Co', 'Cr', 'Fe', 'Ni', 'Mn', 'Cu'}

def fracciones(formula):
    m = parsear_formula(formula); t = sum(m.values())
    return {e: n / t for e, n in m.items()}

def sistema(x):
    if sum(v for e, v in x.items() if e in REFRACTARIOS) >= 0.5: return 'Refractaria'
    if sum(v for e, v in x.items() if e in METALES_3D) >= 0.5: return 'Metales 3d'
    return 'Mixta'

def familia_fase(texto):
    if pd.isna(texto): return 'Sin dato'
    tk = str(texto).replace(' ', '').upper().split('+')
    fcc = any('FCC' in t for t in tk); bcc = any('BCC' in t or t == 'B2' for t in tk); hcp = any('HCP' in t for t in tk)
    if fcc and bcc: return 'FCC+BCC'
    if fcc and not hcp: return 'FCC'
    if bcc and not hcp: return 'BCC'
    if hcp and not fcc and not bcc: return 'HCP'
    return 'Otra'

def secundarias(texto):
    if pd.isna(texto): return False
    tk = str(texto).replace(' ', '').upper().split('+')
    return any(not (('FCC' in t) or ('BCC' in t) or t in ('B2', 'L12', 'HCP')) for t in tk)

d['x'] = d['formula'].apply(fracciones)
d['sistema'] = d['x'].apply(sistema)

# Comprobación: clasificación con la lectura literal de la Etapa 2
def fracciones_literal(formula):
    items, _ = _leer(str(formula).replace(' ', ''))
    m = _expandir(items, normalizar=False); t = sum(m.values())
    return {e: n / t for e, n in m.items()}
cambia = d['formula'].apply(lambda f: sistema(fracciones_literal(f))) != d['sistema']
print('Fórmulas con paréntesis:', d.loc[d['formula'].str.contains(r'\('), 'formula'].nunique(),
      '| registros cuyo sistema cambia con la lectura corregida:', int(cambia.sum()))
print('Ejemplo Co25Cr25(FeNi)50:', {e: round(v, 3) for e, v in fracciones('Co25Cr25(FeNi)50').items()})
d['familia'] = d['fases'].apply(familia_fase)
d['secundarias'] = d['fases'].apply(secundarias)

clave = ['formula', 'doi', 'ensayo', 'T', 'proc']
g = d[d['YS_tipo'] == 'exacto'].groupby(clave, dropna=False)['YS']
info = g.agg(n_unidad='size', ys_min='min', ys_max='max')
d = d.merge(info, left_on=clave, right_index=True, how='left')
d['dispersion_unidad'] = (d['ys_max'] - d['ys_min']) / d['ys_min'] * 100

print(pd.crosstab(d['familia'], d['sistema']))
print('\nRegistros en unidades con varios valores de YS:', int((d['n_unidad'] > 1).sum()))

Fórmulas con paréntesis: 35 | registros cuyo sistema cambia con la lectura corregida: 1
Ejemplo Co25Cr25(FeNi)50: {'Co': 0.25, 'Cr': 0.25, 'Fe': 0.25, 'Ni': 0.25}
sistema   Metales 3d  Mixta  Refractaria
familia                                 
BCC              144      2          618
FCC              382      8            4
FCC+BCC          207      0            0
HCP                0      0            4
Otra              13      4           16
Sin dato         120     12           11

Registros en unidades con varios valores de YS: 242


### 3.1 Fórmulas distintas frente a composiciones distintas

En la Etapa 1 se contaron 672 fórmulas distintas. Algunas describen la misma aleación escrita en otro orden o en otra notación
(por ejemplo, `CoCrFeNi`, `NiCoFeCr` y `Co25Cr25Fe25Ni25`). Se comparan las composiciones atómicas redondeadas a 0.1 %.

In [4]:
d['clave_comp'] = d['x'].apply(lambda x: tuple(sorted((e, round(v, 3)) for e, v in x.items())))
escrituras = d.groupby('clave_comp')['formula'].unique()
varias = escrituras[escrituras.apply(len) > 1]
print('Fórmulas distintas:', d['formula'].nunique(), '| composiciones distintas:', d['clave_comp'].nunique())
print('Composiciones escritas de más de una forma:', len(varias))
for f in varias.head(6): print('  ', list(f))

Fórmulas distintas: 672 | composiciones distintas: 631
Composiciones escritas de más de una forma: 30
   ['Al0.10CoCrFeMnNi', 'FeCoCrNiMnAl0.1']
   ['FeCoCrNiMnAl0.3', 'Al0.3CoCrFeMnNi']
   ['Al0.5CoCrCuFeNi', 'Al9Co18Cr18Cu18Fe18Ni18']
   ['Al0.5CoCrFeNiTi0.5', 'CrCoFeNiAl0.5Ti0.5']
   ['Al0.5CoCrFeNi', 'CoCrFeNiAl0.5']
   ['AlCoCrCuFeNiTi', 'AlCoCuFeNiCrTi']


**Interpretación.** Las 672 fórmulas corresponden a 631 composiciones distintas: 30 composiciones aparecen escritas de dos o más formas. Para contar aleaciones o reunir los registros de una misma aleación, la herramienta usa la composición y no el texto de la fórmula.

## 4. Parámetros de composición

Para cada aleación se calculan, a partir de sus fracciones atómicas $x_i$:

$$\Delta S_{mix} = -R\sum x_i \ln x_i \qquad \delta = 100\sqrt{\sum x_i\left(1-\frac{r_i}{\bar r}\right)^2} \qquad VEC = \sum x_i\,VEC_i$$

$$\rho_{ROM} = \frac{\sum x_i A_i}{\sum x_i A_i/\rho_i}$$

- **ΔS<sub>mix</sub>**: entropía configuracional ideal. Se considera *alta entropía* cuando es ≥ 1.5R.
- **δ**: diferencia de tamaño atómico (%). Las soluciones sólidas simples suelen formarse con δ ≤ 6.6 % (Zhang et al., 2008; Yang y Zhang, 2012).
- **VEC**: concentración de electrones de valencia. Regla empírica de Guo et al. (2011): VEC ≥ 8 favorece FCC, VEC < 6.87 favorece BCC y entre ambos se esperan mezclas FCC+BCC.

Los radios atómicos (radio metálico para coordinación 12, en pm), las densidades y los VEC de cada elemento son **valores
tabulados aproximados**; otros autores usan tablas ligeramente distintas, por lo que δ puede variar unas décimas.

In [5]:
#            masa atómica, densidad (g/cm³), VEC, radio (pm)
ELEMENTOS = {
    'Al': (26.982, 2.70, 3, 143), 'Co': (58.933, 8.90, 9, 125), 'Cr': (51.996, 7.19, 6, 128),
    'Cu': (63.546, 8.96, 11, 128), 'Fe': (55.845, 7.87, 8, 126), 'Mn': (54.938, 7.21, 7, 127),
    'Ni': (58.693, 8.91, 10, 124), 'Ti': (47.867, 4.51, 4, 147), 'V': (50.942, 6.11, 5, 134),
    'Zr': (91.224, 6.51, 4, 160), 'Nb': (92.906, 8.57, 5, 146), 'Mo': (95.95, 10.28, 6, 139),
    'Hf': (178.49, 13.31, 4, 159), 'Ta': (180.948, 16.69, 5, 146), 'W': (183.84, 19.25, 6, 139),
    'Re': (186.207, 21.02, 7, 137), 'Si': (28.086, 2.33, 4, 115), 'C': (12.011, 2.27, 4, 77),
    'B': (10.81, 2.34, 3, 82), 'Zn': (65.38, 7.14, 12, 139), 'Sn': (118.71, 7.29, 14, 158),
    'Mg': (24.305, 1.74, 2, 160), 'Li': (6.94, 0.534, 1, 152), 'Y': (88.906, 4.47, 3, 180),
    'Ca': (40.078, 1.55, 2, 197), 'Pd': (106.42, 12.02, 10, 137), 'Ag': (107.868, 10.49, 11, 144),
    'Sc': (44.956, 2.99, 3, 164), 'Nd': (144.242, 7.01, 3, 182), 'Ga': (69.723, 5.91, 13, 135),
}
R = 8.314

def parametros(x):
    masa = sum(v * ELEMENTOS[e][0] for e, v in x.items())
    vol = sum(v * ELEMENTOS[e][0] / ELEMENTOS[e][1] for e, v in x.items())
    rbar = sum(v * ELEMENTOS[e][3] for e, v in x.items())
    return pd.Series({
        'n_el': len(x),
        'Smix_R': -sum(v * np.log(v) for v in x.values() if v > 0),
        'delta': 100 * np.sqrt(sum(v * (1 - ELEMENTOS[e][3] / rbar) ** 2 for e, v in x.items())),
        'VEC': sum(v * ELEMENTOS[e][2] for e, v in x.items()),
        'rho_calc': masa / vol,
    })

faltan = {e for x in d['x'] for e in x} - set(ELEMENTOS)
print('Elementos sin datos en la tabla:', faltan or 'ninguno')
d = pd.concat([d, d['x'].apply(parametros)], axis=1)
d['Smix'] = d['Smix_R'] * R
print(d[['Smix_R', 'delta', 'VEC', 'rho_calc']].describe().round(2))

Elementos sin datos en la tabla: ninguno
        Smix_R    delta      VEC  rho_calc
count  1545.00  1545.00  1545.00   1545.00
mean      1.55     4.52     6.38      7.95
std       0.19     2.23     1.64      1.76
min       0.78     0.32     1.77      1.38
25%       1.39     3.34     4.61      6.85
50%       1.59     4.40     6.79      7.74
75%       1.61     5.61     7.88      8.54
max       2.20    19.50    10.00     13.65


### 4.1 Comprobación: densidad calculada frente a la densidad ROM del dataset

Si la tabla de elementos es correcta, la densidad por regla de mezclas calculada aquí debe coincidir con la que reporta el
dataset (614 registros).

In [6]:
comp = d.dropna(subset=['rho_rom'])
err = (comp['rho_calc'] - comp['rho_rom']) / comp['rho_rom'] * 100
print('Registros comparados:', len(comp))
print('Error relativo (%): mediana', round(err.median(), 2), '| percentil 5–95:',
      round(err.quantile(.05), 2), 'a', round(err.quantile(.95), 2), '| |error| > 5 %:', int((err.abs() > 5).sum()))
print(comp.assign(err=err).loc[err.abs() > 5, ['formula', 'rho_rom', 'rho_calc', 'err']].drop_duplicates('formula').round(2).head(10).to_string())

Registros comparados: 614
Error relativo (%): mediana -0.02 | percentil 5–95: -3.37 a 0.87 | |error| > 5 %: 18
                 formula  rho_rom  rho_calc    err
72        AlC0.5CoCrFeNi      6.8      6.43  -5.40
73        AlC1.5CoCrFeNi      7.0      5.95 -15.02
74           AlCCoCrFeNi      6.9      6.18 -10.49
101           AlCoCrFeNi      5.7      6.73  18.01
173  Al0.5B0.6CoCrCuFeNi      7.7      7.29  -5.35
174     Al0.5BCoCrCuFeNi      7.8      7.08  -9.21
328         HfNbSi0.5TiV      8.6      7.81  -9.24
396       (CuMnNi)90Al10      8.1      7.56  -6.68
407        Al2CoCrCuFeNi      6.7      6.28  -6.27


**Interpretación.** La mediana del error es −0.02 % y el 90 % central de los registros queda entre −3.4 % y 0.9 %, así que la tabla de elementos reproduce la densidad ROM del dataset. La comparación también respalda la lectura corregida de los paréntesis: con la lectura literal de la Etapa 2, la serie `Al(CoCrCuFeMnNiTiV)` llegaba a diferir 33 %, y con la corregida coincide. En 18 registros la diferencia supera 5 %. Varios contienen carbono o boro, cuya densidad depende de la forma alotrópica que se suponga. Otros parecen errores del dataset: por ejemplo, para `AlCoCrFeNi` el dataset reporta 5.7 g/cm³ y la regla de mezclas da 6.73 g/cm³. Estos registros reciben un aviso de calidad, y la herramienta usa la densidad calculada.

### 4.2 ¿Qué tan bien predice el VEC la fase reportada?

Se compara la regla de Guo et al. con la familia de fase reportada, usando una sola fila por aleación y artículo.

In [7]:
def fase_vec(v):
    return 'FCC' if v >= 8 else ('BCC' if v < 6.87 else 'FCC+BCC')

d['fase_VEC'] = d['VEC'].apply(fase_vec)
u = d[d['familia'].isin(['FCC', 'BCC', 'FCC+BCC'])].drop_duplicates(['formula', 'doi', 'familia'])
acierto = (u['fase_VEC'] == u['familia'])
tabla_vec = pd.crosstab(u['familia'], u['fase_VEC'], margins=True)
print('Aleaciones (fórmula + artículo) con fase FCC, BCC o FCC+BCC:', len(u))
print(tabla_vec.to_string())
print('\nAcierto global:', round(100 * acierto.mean(), 1), '%')
acierto_fam = acierto.groupby(u['familia']).mean().mul(100).round(1)
print('Acierto por fase reportada (%):', acierto_fam.to_dict())
print('Acierto sin fases secundarias:', round(100 * acierto[~u['secundarias']].mean(), 1), '%')

Aleaciones (fórmula + artículo) con fase FCC, BCC o FCC+BCC: 721
fase_VEC  BCC  FCC  FCC+BCC  All
familia                         
BCC       284    1       74  359
FCC        11  141       74  226
FCC+BCC     5   20      111  136
All       300  162      259  721

Acierto global: 74.3 %
Acierto por fase reportada (%): {'BCC': 79.1, 'FCC': 62.4, 'FCC+BCC': 81.6}
Acierto sin fases secundarias: 76.8 %


**Interpretación.** La regla del VEC acierta en el 74.3 % de las 721 combinaciones de aleación y artículo con fase FCC, BCC o FCC+BCC. Acierta más en BCC (79.1 %) y FCC+BCC (81.6 %) que en FCC (62.4 %): 74 aleaciones FCC tienen un VEC entre 6.87 y 8, donde la regla predice una mezcla. Es una regla útil para orientar el diseño, pero no sustituye la caracterización, y por eso la herramienta muestra la predicción junto a la fase reportada.

## 5. Indicador de calidad del dato

Los datos provienen de muchos artículos transcritos por quienes compilaron la base. La herramienta no oculta los registros
dudosos: los muestra con avisos. Cada aviso tiene un peso y la suma define el nivel.

| Aviso | Condición | Peso |
|---|---|---|
| Elongación plástica mayor que la total | físicamente imposible (Etapa 1) | 1 |
| Módulo de Young experimental < 50 GPa | muy bajo para estas aleaciones (Etapa 1) | 1 |
| Razón YS/HV fuera de 1–6 | lejos de σ<sub>y</sub> ≈ HV/3 (Etapa 1) | 1 |
| Carbono medido distinto del nominal | diferencia > 20 % (Etapa 1) | 1 |
| Elongación reportada como límite o rango | ">50", "<1", "1-2" | 0.5 |
| Elongación de 50 % en compresión | posible deformación máxima del ensayo (Etapa 1) | 0.5 |
| Densidad ROM del dataset distinta de la calculada | diferencia > 5 % (sección 4.1) | 0.5 |
| YS calculado o extraído de gráficas | comentario de los compiladores | 0.5 |
| Procesamiento corregido por los compiladores | comentario | 0.5 |
| Misma etiqueta con YS distinto | unidad con varios registros y dispersión > 20 % (Etapa 2) | 0.5 |

Además se muestran **notas** sin peso, que ayudan a interpretar el dato: la elongación en compresión es deformación máxima,
no elongación a fractura, y algunos registros no reportan la fase.

**Nivel:** *Alta* sin avisos; *Media* con suma 0.5 o 1; *Baja* con suma ≥ 1.5.

In [8]:
com = d['comentario'].fillna('').str.lower()
razon = d['YS'].where(d['YS_tipo'] == 'exacto') / d['HV'].where(d['HV_tipo'] == 'exacto')
# Carbono nominal en % en peso a partir de la fórmula, frente al medido (wppm / 10^4)
C_nominal = d['x'].apply(lambda x: 100 * x.get('C', 0) * ELEMENTOS['C'][0] / sum(v * ELEMENTOS[e][0] for e, v in x.items()))
C_medido = d['C_wppm'] / 1e4

AVISOS = [
    ('pl_mayor_total', 'La elongación plástica es mayor que la total', 1, d['Elp'] > d['El']),
    ('E_bajo', 'Módulo de Young experimental menor que 50 GPa', 1, d['Eexp'] < 50),
    ('ys_hv', 'Razón YS/HV fuera de 1 a 6 (se espera cerca de 3)', 1, (razon < 1) | (razon > 6)),
    ('carbono', 'El carbono medido difiere más de 20 % del nominal de la fórmula', 1, (C_nominal > 0) & ((C_medido - C_nominal).abs() / C_nominal > 0.2)),
    ('el_limite', 'Elongación reportada como límite o rango, no como valor exacto', 0.5, d['El_tipo'].isin(['limite_inferior', 'limite_superior', 'rango'])),
    ('el_50_comp', 'Elongación de 50 % en compresión: probablemente el ensayo se detuvo ahí', 0.5, (d['ensayo'] == 'C') & (d['El'] == 50) & (d['El_tipo'] == 'exacto')),
    ('rho_rom', 'La densidad ROM del dataset difiere más de 5 % de la calculada con la composición', 0.5, ((d['rho_rom'] - d['rho_calc']).abs() / d['rho_calc'] > 0.05)),
    ('ys_derivado', 'YS calculado o extraído de gráficas por los compiladores', 0.5, com.str.contains('calculated|stress strain curves|from plot|off plot|extracted data')),
    ('proc_corregido', 'Procesamiento corregido por los compiladores', 0.5, com.str.contains('wrong processing|changed processing')),
    ('unidad_dispersa', 'Hay otros registros con la misma aleación, artículo, ensayo, temperatura y procesamiento, y su YS difiere más de 20 %', 0.5, (d['n_unidad'] > 1) & (d['dispersion_unidad'] > 20)),
]
NOTAS = [
    ('el_compresion', 'En compresión, la elongación es la deformación máxima del ensayo y no se compara con la de tensión', (d['ensayo'] == 'C') & d['El'].notna()),
    ('sin_fase', 'El artículo no reporta la fase', d['familia'] == 'Sin dato'),
    ('uts_compresion', 'En compresión, el UTS es la resistencia máxima a compresión', (d['ensayo'] == 'C') & d['UTS'].notna()),
]

d['avisos'] = [[] for _ in range(len(d))]
d['notas'] = [[] for _ in range(len(d))]
puntaje = pd.Series(0.0, index=d.index)
for clave_av, texto, peso, mask in AVISOS:
    mask = mask.fillna(False).astype(bool)
    for i in d.index[mask]: d.at[i, 'avisos'].append(clave_av)
    puntaje += mask * peso
for clave_n, texto, mask in NOTAS:
    mask = mask.fillna(False).astype(bool)
    for i in d.index[mask]: d.at[i, 'notas'].append(clave_n)
d['calidad'] = np.select([puntaje >= 1.5, puntaje > 0], ['Baja', 'Media'], 'Alta')

print(d['calidad'].value_counts().reindex(['Alta', 'Media', 'Baja']).to_string())
print('\nFrecuencia de cada aviso:')
print(pd.Series({t: int(m.fillna(False).sum()) for _, t, _, m in AVISOS}).to_string())

calidad
Alta     1225
Media     310
Baja       10

Frecuencia de cada aviso:
La elongación plástica es mayor que la total                                                                               4
Módulo de Young experimental menor que 50 GPa                                                                              7
Razón YS/HV fuera de 1 a 6 (se espera cerca de 3)                                                                         11
El carbono medido difiere más de 20 % del nominal de la fórmula                                                            3
Elongación reportada como límite o rango, no como valor exacto                                                            57
Elongación de 50 % en compresión: probablemente el ensayo se detuvo ahí                                                   27
La densidad ROM del dataset difiere más de 5 % de la calculada con la composición                                         19
YS calculado o extraído de gráficas por los comp

## 6. Exportación para la página

Se exportan las columnas que usa la herramienta, con nombres cortos y `null` en lugar de `NaN`. `datos.js` define las
variables globales `META` y `DATOS`, de modo que la página funciona en GitHub Pages y también al abrir `index.html`
directamente.

In [9]:
def limpio(v):
    return None if (v is None or (isinstance(v, float) and np.isnan(v))) else v

salida = pd.DataFrame({
    'id': np.arange(len(d)), 'f': d['formula'], 'doi': d['doi'], 'fases': d['fases'].fillna(''),
    'fam': d['familia'], 'sec': d['secundarias'], 'sis': d['sistema'], 'proc': d['proc'].fillna('Sin dato'),
    'ens': d['ensayo'].map({'C': 'Compresión', 'T': 'Tensión'}).fillna('Sin dato'), 'T': d['T'],
    'YS': d['YS'], 'YSi': d['YS_inc'], 'UTS': d['UTS'], 'UTSi': d['UTS_inc'],
    'El': d['El'], 'Elt': d['El_tipo'], 'Elp': d['Elp'], 'HV': d['HV'], 'Eexp': d['Eexp'], 'Erom': d['E_rom'],
    'rho': d['rho_calc'].round(3), 'rhoR': d['rho_rom'], 'rhoE': d['rho_exp'], 'grano': d['grano'], 'O': d['O'],
    'n': d['n_el'].astype(int), 'S': d['Smix_R'].round(3), 'dl': d['delta'].round(2), 'VEC': d['VEC'].round(3),
    'fv': d['fase_VEC'], 'YSr': (d['YS'] / d['rho_calc']).round(1),
    'x': d['x'].apply(lambda x: {e: round(v, 4) for e, v in sorted(x.items(), key=lambda kv: -kv[1])}),
    'com': d['comentario'].fillna(''), 'cal': d['calidad'], 'av': d['avisos'], 'no': d['notas'],
})
registros = json.loads(salida.to_json(orient='records', force_ascii=False))
meta = {
    'fuente': 'Borg et al. (2020), Scientific Data 7, 430 — Dataset 3 del curso',
    'n': len(registros),
    'elementos': {e: {'A': v[0], 'rho': v[1], 'VEC': v[2], 'r': v[3]} for e, v in ELEMENTOS.items()},
    'acierto_vec': round(float(100 * acierto.mean()), 1),
    'acierto_vec_fam': {k: float(v) for k, v in acierto_fam.items()},
    'n_vec': int(len(u)),
    'avisos': {k: {'texto': t, 'peso': p} for k, t, p, _ in AVISOS},
    'notas': {k: t for k, t, _ in NOTAS},
}
with open('datos.js', 'w', encoding='utf-8') as fh:
    fh.write('// Generado por preparar_datos.ipynb. No editar a mano.\n')
    fh.write('const META = ' + json.dumps(meta, ensure_ascii=False) + ';\n')
    fh.write('const DATOS = ' + json.dumps(registros, ensure_ascii=False, separators=(',', ':')) + ';\n')

print('Registros exportados:', len(registros))
print('Tamaño de datos.js:', round(os.path.getsize('datos.js') / 1024), 'KB')

Registros exportados: 1545
Tamaño de datos.js: 790 KB


## 7. Descarga del archivo (solo en Colab)

In [10]:
try:
    from google.colab import files
    files.download('datos.js')
except ImportError:
    print('Fuera de Colab: datos.js quedó en la carpeta actual.')

Fuera de Colab: datos.js quedó en la carpeta actual.
